In [1]:
%load_ext autoreload
%autoreload 2

import json
import os
import time

import matplotlib.pyplot as plt
import numpy as np

from polyergalio.encoders.pipeline import UniversalPipeline
from polyergalio.encoders import TextProcessor, Tokenizer, SentencePieceTokenizer
from polyergalio.encoders.tokenizer import TokenSequenceBuilder, fit_tokenizer
from polyergalio.generators.data_generators import token_accuracy

from polyergalio.models.constants import ClassificationTask
from polyergalio.models.embedding.embedding import TextEmbedding
from polyergalio.models.heads import LowRankBottleNeck
from polyergalio.models.embedding.positional import RopeEmbedding
from polyergalio.models.network import Network
from polyergalio.models.layers import (
    DropoutLayer,
    FullyConnectedLayer,
    RMSNormLayer,
    MixtureOfExperts,
    SpectreAttention,
    LatentSum,
    MaskGather
)

from polyergalio.models.model_loss import CrossEntropyLoss, DecisionLoss, SparseCrossEntropyLoss
from polyergalio.models.optimizers import Adam




ModuleNotFoundError: No module named 'polyergalio.models.layers.decomposition_layers'

In [ ]:
# THIS IS SUPER KLUDGEY -- clean it up...

In [ ]:
# TOKENIZER
TARGET_VOCAB_SIZE = 55_000
SEQUENCE_LENGTH = 500

corpus_path = f"../data/decision_data"
all_json = []

for fn in os.listdir(corpus_path):
    if fn.endswith(".jsonl"):
        with open(f"{corpus_path}/{fn}", "r") as handle:
            for line in handle:
                try:
                    all_json.append(json.loads(line))
                except:
                    continue
    elif fn.endswith(".json"):
        with open(f"{corpus_path}/{fn}", "r") as handle:
            try:
                one_dict = json.loads(line)
                if "options" in one_dict.keys():
                    all_json.append(one_dict)
            except:
                continue



In [ ]:
print(len(all_json))

In [ ]:
# flatten out all of our strings for tokenizer fitting --
line_by_line = []
for sample in all_json:
    concat = ""
    try:
        for key, value in sample.items():
            concat += f"{key}: {value}\n"
            line_by_line.append(concat)
        # if "options" in concat:
        #     line_by_line.append(concat)
        # else:
        #     print('skipped')
    except AttributeError:
        print(sample)


In [ ]:
print(line_by_line[5])
print(line_by_line[-10])

In [ ]:
# model_path = fit_tokenizer(
#     corpus=line_by_line,
#     model_prefix=f"decision_tokenizer",
#     vocab_size=TARGET_VOCAB_SIZE,
#     hard_vocab_limit=False,
# )
#
tokenizer = SentencePieceTokenizer("decision_tokenizer.model")

In [ ]:
# tokenizer
feature_encoder = UniversalPipeline(name="feature_encoder")

In [ ]:
# # Starting -------------
from polyergalio.encoders.text_encoders import DistortionTask
tasks = (DistortionTask.CLOZE,
         DistortionTask.REPLACED_TOKEN,
         DistortionTask.TOKEN_DELETION,
         )
#
# feature_encoder = UniversalPipeline(name="feature_encoder")
# text_processor = TextProcessor(tokenizer,
#                                max_length=SEQUENCE_LENGTH,
#                                replace_prob=0.12,
#                                max_span_length=SEQUENCE_LENGTH,
#                                # tasks=tasks,
#                                pool_size=50_000,
#                                )
#
# feature_encoder.connect(text_processor, name="tokenizer")
# feature_encoder["tokenizer"].fit(line_by_line)
# feature_encoder.serialize("spectre_decision_feature.pipeline")
# feature_encoder = UniversalPipeline(name="feature_encoder")

# text_processor.fit(line_by_line)

In [ ]:
# feature_encoder.serialize("spectre_decision_feature.pipeline")
print("feature serialized")

In [ ]:
# ---------------------------------
# LOADING TEXT PIPELINE --
feature_encoder = UniversalPipeline(name="feature_encoder")
feature_encoder = feature_encoder.deserialize("spectre_decision_feature.pipeline")

print(feature_encoder["tokenizer"])
feature_encoder["tokenizer"].max_length = SEQUENCE_LENGTH
feature_encoder["tokenizer"].tokenizer.max_length = SEQUENCE_LENGTH
feature_encoder["tokenizer"].tokenizer.sp.max_sentence_length = SEQUENCE_LENGTH

# double-checking ---
feature_encoder["tokenizer"].encode("this")
feature_encoder["tokenizer"].sentence_pool

In [ ]:
HIDDEN_DIM = 960
# FFN_HIDDEN = 750
NUM_HEADS = 12
DROPOUT_PROB = 0.10
PADDING_TOK = feature_encoder["tokenizer"].tokenizer.special_tokens.PAD

net = Network(name="spectre_transformer", input_shape=(SEQUENCE_LENGTH,))
embedding = net.connect(
    TextEmbedding(TARGET_VOCAB_SIZE, HIDDEN_DIM, padding_idx=PADDING_TOK), net.input, name="embedding"
)
positional = net.connect(RopeEmbedding(SEQUENCE_LENGTH, HIDDEN_DIM), embedding, name="positional_emb")

# ------ SpectreTF block
at = net.connect(RMSNormLayer(HIDDEN_DIM), positional, name="prenorm_0")
at = net.connect(SpectreAttention(SEQUENCE_LENGTH,
                                  HIDDEN_DIM,
                                  num_heads=NUM_HEADS,
                                  band_radius=2,
                                  memory_tokens=100,
                                  use_wrm=True), at, name="attention_0")
at = net.connect(FullyConnectedLayer(HIDDEN_DIM, HIDDEN_DIM, "swish"), at, name="ffna_0")
at = net.connect(RMSNormLayer(HIDDEN_DIM), at, name="postnorm_0")
at = net.connect(DropoutLayer(DROPOUT_PROB), at, name="dropout_0")
at = net.connect(LatentSum(), at, positional, name="residual_0")

# downsample = [20, 20, 10, 10, 10, 5, 5, 2, 2, 0, 0, 0, 0, 0, 0]
for block_idx in range(1, 12):
    use_wrm = True
    # ------ SpectreTF blocks
    at = net.connect(RMSNormLayer(HIDDEN_DIM), at, name=f"prenorm_{block_idx}")
    at = net.connect(SpectreAttention(SEQUENCE_LENGTH,
                                      HIDDEN_DIM,
                                      num_heads=NUM_HEADS,
                                      band_radius=0,
                                      memory_tokens=128,
                                      use_wrm=use_wrm
                                      ), at, name=f"attention_{block_idx}")

    if (block_idx % 2) == 0:
        at = net.connect(MixtureOfExperts(input_dim=HIDDEN_DIM,
                                          upscale_dim=HIDDEN_DIM,
                                          hidden_dim=HIDDEN_DIM,
                                          num_shared_experts=1,
                                          num_routed_experts=16,
                                          activation_type="swish",
                                          top_k=2,
                                          num_groups=8,
                                          top_groups=4,
                                          routed_scaling=1.2), at, name=f"ffnmoe_{block_idx}")

    else:
        at = net.connect(FullyConnectedLayer(HIDDEN_DIM, HIDDEN_DIM, "swish"), at, name=f"ffna_{block_idx}")

    at = net.connect(DropoutLayer(DROPOUT_PROB), at, name=f"dropout_{block_idx}")
    at = net.connect(LatentSum(), at, positional, name=f"residual_{block_idx}")


at = net.connect(DropoutLayer(DROPOUT_PROB), at, name=f"dropout_trunk")
at = net.connect(MixtureOfExperts(input_dim=HIDDEN_DIM,
                                      upscale_dim=int(1.25 * HIDDEN_DIM),
                                      hidden_dim=HIDDEN_DIM,
                                      num_shared_experts=2,
                                      num_routed_experts=32,
                                      activation_type="swish",
                                      top_k=4,
                                      num_groups=8,
                                      top_groups=4,
                                      routed_scaling=1.2), at, name=f"moetrunk")

gathered = net.connect(MaskGather(), at, name="mask_gather")
net.output = net.connect(
    LowRankBottleNeck(HIDDEN_DIM, HIDDEN_DIM // 4, TARGET_VOCAB_SIZE), gathered, name="mlm_head"
    # FullyConnectedLayer(HIDDEN_DIM, TARGET_VOCAB_SIZE, "linear", is_output=True), gathered, name="mlm_head"
)

net.recast(data_type=np.float32, complex_data_type=np.complex64)

In [ ]:
print(net.summary())

In [ ]:
from polyergalio.visuals.nnet_visuals import plot_network

print("edges:")
for producer, consumer in net.edges():
    print(f"  {producer:10s} -> {consumer}")
_temp = feature_encoder["tokenizer"].encode(line_by_line[5:7])
print(net.summary(_temp["text"]))
plot_network(net)

In [ ]:
print(net.timing_summary(top=10))

In [ ]:
def mlm_accuracy(net: Network, processor: TextProcessor, texts: list[str]) -> float:
    """Accuracy at masked positions of a cloze batch, with the network in eval mode."""
    batch = processor.distort_batch(texts, "cloze")
    net.eval()
    logits = net.forward(batch["input_ids"], mask=batch["attention_mask"], target_mask=batch["target_mask"])
    return token_accuracy(np.argmax(logits, axis=-1), batch["labels"][batch["target_mask"]])


print(net)

In [ ]:
# feature_encoder pipeline needs to have visibility into these subprocesses:
data = feature_encoder["tokenizer"].distort_batch(line_by_line[-5:])
data.keys()

In [ ]:
held_out = []
training = []
samples = np.random.choice([True, False], replace=True, size=len(line_by_line), p=[0.05, 0.95])
for idx, hold in enumerate(samples):
    if hold == True:
        held_out.append(line_by_line[idx])
    else:
        training.append(line_by_line[idx])

print(len(held_out))
print(len(training))

In [ ]:
len(feature_encoder["tokenizer"].sentence_pool)

In [ ]:
# deserialize

net = net.deserialize("spectre_nnet.ergalio")

In [ ]:
import random
%pdb on
BATCH_SIZE = 5
LR = 3e-4
EPOCHS = 10
LOG_EVERY = 20
# _targets = np.eye(TARGET_VOCAB_SIZE)

all_loss = []
loss_fn = SparseCrossEntropyLoss()
optimizer = Adam(LR)

for epoch in range(0, EPOCHS):
    np.random.shuffle(training)
    np.random.shuffle(training)
    optimizer.learning_rate = LR * (0.92 ** epoch)
    print(f"training will take {len(training) / BATCH_SIZE} steps")

    for step, start_idx in enumerate(range(150, len(training), BATCH_SIZE)):
        net.train()
        _mini_batch = training[start_idx:start_idx + BATCH_SIZE]

        _tok_batch = feature_encoder["tokenizer"].distort_batch(_mini_batch)

        tokens, labels = _tok_batch["input_ids"], _tok_batch["labels"]
        attention_mask, target_mask = _tok_batch["attention_mask"], _tok_batch["target_mask"]
        targets = labels[target_mask]

        st = time.time()
        logits = net.forward(tokens, mask=attention_mask, target_mask=target_mask)
        loss = loss_fn(logits, labels[target_mask])
        stt = time.time() - st

        net.backward(loss_fn.backward())
        optimizer.step(net)
        net.zero_gradients()
        all_loss.append(loss.item())

        if step % LOG_EVERY == 0:
            print(_tok_batch["task"])
            print(f"mlm step {step:4d}  loss {loss:.4f}")
            print(f"took {stt}")
            plt.plot(all_loss)
            plt.show()
            print(f"masked-token accuracy after training:  {mlm_accuracy(net,
                                                                         feature_encoder["tokenizer"],
                                                                         random.choices(held_out, k=BATCH_SIZE)
                                                                         ):.3f}")

            net.serialize("spectre_nnet_mini.ergalio")

    net.serialize("spectre_nnet.ergalio")
    print(f"masked-token accuracy after training:  {mlm_accuracy(net,
                                                                 feature_encoder["tokenizer"],
                                                                 random.choices(held_out, k=BATCH_SIZE)
                                                                 ):.3f}")


In [ ]:
print(_tok_batch["task"])
print(f"mlm step {step:4d}  loss {loss:.4f}")
print(f"took {stt}")
plt.plot(all_loss)
plt.show()
print(f"masked-token accuracy after training:  {mlm_accuracy(net,
                                                             feature_encoder["tokenizer"],
                                                             random.choices(held_out, k=BATCH_SIZE)
                                                             ):.3f}")

net.serialize("spectre_nnet.ergalio")

In [ ]:
# deserialize
net = net.deserialize("spectre_nnet.ergalio")

In [ ]:
HIDDEN_DIM = 720
net.output = net.connect(DecisionHead(hidden_dim=HIDDEN_DIM,
                                      head_hidden=HIDDEN_DIM,
                                      routed_scaling=1.5,
                                      num_groups=16,
                                      top_groups=8),
                         net.node("residual_11"),
                         name="decision_moe_out")

net.disconnect(net.node("decision_moe"))
net.delete(net.node("decision_moe"))

net.purge()
plot_network(net)

In [ ]:
# how to disconnect / reconnect ----------
# pooler = net.connect(PoolingLayer(),
#                      net.node("residual_11"),
#                      name="pooling")
# HIDDEN_DIM=720
#
# net.output = net.connect(DecisionHead(hidden_dim=HIDDEN_DIM,
#                                       head_hidden=HIDDEN_DIM),
#                          net.node("residual_11"),
#                          name="decision_moe")
#
# net.disconnect(net.node("mask_gather"))
# net.delete(net.node("mask_gather"))
# net.disconnect(net.node("mlm_head"))
# net.delete(net.node("mlm_head"))
#
# net.purge()
# plot_network(net)


task_processor = TokenSequenceBuilder(tokenizer)

task_processor.build_decision_batch(samples, max_length=None)

## yields :
token_ids (batch, sequence_length)
mask (batch, sequence_length) — the encoder's attention mask, token_ids != PAD
marker_pos -> marker token positions (the start of a span)
marker_end -> marker_end token positions (the ending of a span)
token_mask (batch, options) — DecisionHead's own kwargs
decisiontypes (batch,) - one Enum per task
answers (batch,), included only when every sample carries an answer

In [ ]:

sample = [
    {"id": 0,
     "decisiontype": "choice",
     "instructions": "What should be done with this inventory item?",
     "options": ["reorder", "discount", "hold", "discontinue"],

     "state": {"sku": "SKU-4122", "item": "HDMI cable", "stock": 324, "weekly_sales": 1, "days_to_expiry": None},
     "answer": 3,
     "option": "discontinue",
     "record": {"sku": "SKU-4122", "item": "HDMI cable", "stock": 324, "weekly_sales": 1, "days_to_expiry": None}
     },
    {"id": 0,
     "decisiontype": "choice",
     "instructions": "What should be done with this inventory item?",
     "options": ["reorder", "discount", "hold", "discontinue"],

     "state": {"sku": "SKU-4122", "item": "HDMI cable", "stock": 324, "weekly_sales": 1, "days_to_expiry": None},
     "answer": 3,
     "option": "discontinue",
     "record": {"sku": "SKU-4122", "item": "HDMI cable", "stock": 324, "weekly_sales": 1, "days_to_expiry": None}
     }
]

task_processor = TokenSequenceBuilder(feature_encoder["tokenizer"].tokenizer)

# task_processor.build_decision_batch
batch = task_processor.build_decision_batch(sample, max_length=512)
# target = (batch["answer"], batch["option"])


print("edges:")
for producer, consumer in net.edges():
    print(f"  {producer:10s} -> {consumer}")

st = time.time()
logits = net.forward(batch["token_ids"],
                     mask=batch["mask"],
                     marker_pos=batch["marker_pos"],
                     marker_end=batch["marker_end"],
                     token_mask=batch["token_mask"],
                     decisiontypes=batch["decisiontypes"])
print(f'naive forward took {(time.time() - st) / len(sample)} per sample row')
print(net.timing_summary(top=10))


In [ ]:
batch

In [ ]:
loss = DecisionLoss(ordinal_weight=0.25)(logits,
                                         batch["answers"],
                                         batch["token_mask"],
                                         batch["decisiontypes"])

print(loss)
print(logits)

In [ ]:
print(net)
# 159_260_541

In [ ]:
held_out = []
training = []
samples = np.random.choice([True, False], replace=True, size=len(all_json), p=[0.05, 0.95])

for idx, hold in enumerate(samples):
    if hold == True:
        held_out.append(all_json[idx])
    else:
        training.append(all_json[idx])

print(len(held_out))
print(len(training))

In [ ]:
net = NeuralNetwork()
net = net.deserialize("spectre_nnet_decider.ergalio")

In [ ]:
net.layers

In [ ]:


BATCH_SIZE = 5
LR = 2e-4
ACT_WEIGHT = 0.01
EPOCHS = 10
LOG_EVERY = 50

all_loss, all_act_loss = [], []
loss_fn = DecisionLoss(ordinal_weight=0.25)
task_processor = TokenSequenceBuilder(feature_encoder["tokenizer"].tokenizer)
optimizer = Adam(LR)

# held_out ----
test_accuracy = []

for epoch in range(0, EPOCHS):
    np.random.shuffle(training)
    optimizer.learning_rate = LR * (0.92 ** epoch)
    net.node("decision_moe_out").component.act_weight = ACT_WEIGHT * (1.2 ** epoch)
    print(f"training will take {len(training) / BATCH_SIZE} steps")

    net.eval()
    net.node("decision_moe_out").component.train()

    for step, start_idx in enumerate(range(0, len(training), BATCH_SIZE)):
        net.zero_gradients()
        _mini_batch = training[start_idx:start_idx + BATCH_SIZE]

        _tok_batch = task_processor.build_decision_batch(
            _mini_batch,
            max_length=SEQUENCE_LENGTH)
        # target = (sample["answer"], sample["option"])

        logits = net.forward(_tok_batch["token_ids"],
                             mask=_tok_batch["mask"],
                             marker_pos=_tok_batch["marker_pos"],
                             marker_end=_tok_batch["marker_end"],
                             token_mask=_tok_batch["token_mask"],
                             decisiontypes=_tok_batch["decisiontypes"])

        act = net.node("decision_moe_out").component.score_act(_tok_batch["answers"])

        loss = loss_fn(logits,
                       _tok_batch["answers"],
                       _tok_batch["token_mask"],
                       _tok_batch["decisiontypes"])

        net.backward(loss_fn.backward())
        optimizer.step(net.layers)

        all_loss.append(loss)
        all_act_loss.append(act)

        this_step = []
        timings = []
        if (step + 1) % LOG_EVERY == 0:
            plt.clf()
            for batch in range(0, len(held_out), BATCH_SIZE):
                st = time.time()
                _tok_batch = task_processor.build_decision_batch(
                    held_out[batch:batch + BATCH_SIZE],
                    # random.choices(held_out, k=BATCH_SIZE),
                    max_length=SEQUENCE_LENGTH)

                net.zero_gradients()
                logits = net.forward(_tok_batch["token_ids"],
                                     mask=_tok_batch["mask"],
                                     marker_pos=_tok_batch["marker_pos"],
                                     marker_end=_tok_batch["marker_end"],
                                     token_mask=_tok_batch["token_mask"],
                                     decisiontypes=_tok_batch["decisiontypes"])

                act = net.node("decision_moe_out").component.score_act(_tok_batch["answers"])
                timings.append(st - time.time())
                loss = loss_fn(logits,
                               _tok_batch["answers"],
                               _tok_batch["token_mask"],
                               _tok_batch["decisiontypes"])

                classes = np.argmax(logits, axis=-1)
                acc = np.sum(classes == _tok_batch["answers"]) / len(classes)
                this_step.append(acc)

            print("decision_types: ", _tok_batch["decisiontypes"])
            print("answer_targets: ", _tok_batch["answers"])
            print("predicted:      ", np.argmax(logits, axis=-1))
            print("esc? : ", net.node("decision_moe_out").component.escalate(0.5))
            print(f"Losses for step {step:4d} \n\t- Decision loss {loss:.4f} \n\t- Confidence loss {act:.4f} ")

            avg = np.mean(timings)
            print(f"mean timings of test dataset per batch of {BATCH_SIZE}: {avg}, \n per sample {avg / BATCH_SIZE}")

            test_accuracy.append(np.mean(this_step))
            print("accuracy:", test_accuracy[-5:])

            plt.figure()
            plt.subplot(3, 1, 1)
            plt.title("Decision Loss")
            plt.plot(all_loss)
            plt.subplot(3, 1, 2)
            plt.title("Confidence/action loss")
            plt.plot(all_act_loss, color="red")
            plt.subplot(3, 1, 3)
            plt.title("test set accuracy")
            plt.plot(test_accuracy, color="green")
            plt.show()

    net.serialize("spectre_nnet_decider.ergalio")


In [ ]:
net.serialize("spectre_nnet_decider.ergalio")